**DETECCIÓN Y LECTURA DE MATRÍCULAS EN VÍDEO CON RT-DETRv2 Y PADDLEOCR-VL**

En este cuaderno se detectan personas y vehículos con RT-DETRv2, se les asigna un identificador con ByteTrack, se localiza la matrícula de cada vehículo con el detector de matrículas (RT-DETRv2 tamaño `s`) y se lee su texto con el modelo de visión-lenguaje `PaddlePaddle/PaddleOCR-VL`.

El flujo de trabajo se estructura en:
1. **Configuración inicial**: Importaciones, clonado de RT-DETR y selección del dispositivo (GPU si está disponible).
2. **Modelo de matrículas**: Carga del detector de matrículas.
3. **Funciones de OCR con VLM**: Carga de PaddleOCR-VL y lectura de matrículas con su confianza.
4. **Parámetros del vídeo y modelo de vehículos**: Parámetros de detección, seguimiento y OCR, y carga del detector de vehículos.
5. **Procesamiento de vídeo**: Detección, seguimiento, localización de matrículas y OCR sobre un vídeo, guardando el vídeo anotado y un CSV.
6. **Evaluación del VLM**: Exact match y acierto por carácter sobre imágenes anotadas, con y sin corrección por posición, y tiempos de lectura.

**CONFIGURACIÓN INICIAL: IMPORTS Y DISPOSITIVO**

Se clona el repositorio de RT-DETR si no existe, se añade su ruta a `sys.path`, se define el identificador del VLM y se selecciona la GPU si PyTorch detecta una con CUDA.

In [2]:
import os, sys, subprocess
from functools import partial
from pathlib import Path
import torch

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

#Cargamos el modelo y ponemos la ruta en memoria
if not os.path.exists("RT-DETR"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lyuwenyu/RT-DETR.git"], check=True)


RTDETR_PATH = os.path.abspath("RT-DETR/rtdetrv2_pytorch")
if RTDETR_PATH not in sys.path:
    sys.path.insert(0, RTDETR_PATH)

from load_rtdetr import load_rtdetr
from utils_matriculas import (
    GT_KEYS,
    cargar_gt,
    crear_detector_rtdetr,
    evaluar_dataset,
    get_ocr_vlm,
    leer_placa_vlm,
    procesar_video_rtdetr,
)

VLM_ID = "PaddlePaddle/PaddleOCR-VL"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**MODELO DE MATRÍCULAS**

Se muestran las configuraciones disponibles en el repositorio y se carga el detector de matrículas (RT-DETRv2 tamaño `s`), con una única clase `plate`, que se aplica sobre el recorte de cada vehículo.

In [3]:
repo = Path("RT-DETR/rtdetrv2_pytorch")

print("Configuraciones RT-DETRv2:")
for f in (repo / "configs/rtdetrv2").glob("*.yml"):
    print(f.name)

print("\nConfiguraciones de matrículas en todo el repositorio:")
for f in repo.rglob("*plate*.yml"):
    print(f)

PLATE_NAME = {0: "matrícula"}
PLATE_CLASS = PLATE_NAME.keys()
PLATE_COLOR = {0: (128, 255, 128)}

model_plate, postprocessor_plate = load_rtdetr(family="v2", size="s", device=DEVICE)

Configuraciones RT-DETRv2:
rtdetrv2_hgnetv2_h_6x_coco.yml
rtdetrv2_hgnetv2_l_6x_coco.yml
rtdetrv2_hgnetv2_x_6x_coco.yml
rtdetrv2_r101vd_6x_coco.yml
rtdetrv2_r18vd_120e_coco.yml
rtdetrv2_r18vd_120e_plates.yml
rtdetrv2_r18vd_120e_voc.yml
rtdetrv2_r18vd_dsp_3x_coco.yml
rtdetrv2_r18vd_sp1_120e_coco.yml
rtdetrv2_r18vd_sp2_120e_coco.yml
rtdetrv2_r18vd_sp3_120e_coco.yml
rtdetrv2_r34vd_120e_coco.yml
rtdetrv2_r34vd_dsp_1x_coco.yml
rtdetrv2_r50vd_6x_coco.yml
rtdetrv2_r50vd_dsp_1x_coco.yml
rtdetrv2_r50vd_m_7x_coco.yml
rtdetrv2_r50vd_m_dsp_3x_coco.yml

Configuraciones de matrículas en todo el repositorio:
RT-DETR\rtdetrv2_pytorch\configs\dataset\plate_detection.yml
RT-DETR\rtdetrv2_pytorch\configs\rtdetrv2\rtdetrv2_r18vd_120e_plates.yml
Cargando RT-DETR familia 'v2', tamaño 's' en cuda


**FUNCIONES DE OCR: PADDLEOCR-VL**

Se carga el VLM una sola vez (`get_ocr_vlm`) y se fija la función de lectura `leer_placa`:
- **`norm`**: Normaliza el texto a mayúsculas alfanuméricas.
- **`recortar`**: Recorta una región con margen porcentual o en píxeles, ajustada a los límites de la imagen.
- **`leer_placa_vlm`**: Reescala el recorte, lo pasa al VLM con el prompt `OCR:` y devuelve el texto y la confianza media de los tokens generados.

Aquí se fijan con `partial` el dispositivo y el modelo ya cargado, para que `leer_placa(crop)` reciba solo el recorte.

En `get_ocr_vlm` el tipo de dato se elige según `DEVICE.type`: `bfloat16` en CUDA y `float32` en CPU. Si `bfloat16` diera un error de tipos en tu GPU, cambia `torch.bfloat16` por `torch.float32` en `utils_matriculas.py`.

In [4]:
# Inicializamos el VLM globalmente: devuelve (modelo, processor), o None si falla la carga
ocr_global = get_ocr_vlm(DEVICE, VLM_ID)

# leer_placa_vlm (utils) recibe el device y el modelo como argumentos: los fijamos aqui
# para conservar la firma leer_placa(crop) -> (texto, confianza)
leer_placa = partial(leer_placa_vlm, device=DEVICE, ocr_inst=ocr_global)

c:\Users\enriq\anaconda3\envs\matriculas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


**PARÁMETROS DEL VÍDEO Y MODELO DE VEHÍCULOS**

Parámetros de detección, seguimiento y OCR sobre vídeo:
- **Clases y colores**: Personas, coches, motos, autobuses y camiones (clases COCO).
- **Recortes de vehículo**: Las matrículas se buscan dentro del recorte de cada vehículo y heredan su `tracker_id`.
- **Lectura OCR**: Frecuencia, número máximo de intentos y confianza mínima para dar una lectura por buena.
- **Patrones de matrícula**: Las expresiones regulares de formatos europeos `PATRONES_MATRICULA` y `matricula_valida` están en `utils_matriculas.py`.

Aquí se carga también el detector de vehículos y personas (RT-DETRv2 tamaño `l`).

In [5]:
CLASS_NAMES = {0: 'persona', 2: 'coche', 3: 'moto', 5: 'guagua', 7: 'camión'}
BASE_CLASS = CLASS_NAMES.keys()
BASE_COLORS = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255), 5: (0, 255, 255), 7: (128, 128, 255)}

# Las matriculas se buscan dentro del recorte de cada vehiculo y heredan su tracker_id
MAX_CROPS = 8          # maximo de vehiculos por frame donde se buscan matriculas
MIN_CROP_SIDE = 48     # px: vehiculos mas pequenos que esto se ignoran
CROP_MARGIN = 0.05     # margen extra alrededor del vehiculo
PLATE_CONF = 0.3      # confianza minima del detector de matriculas
HOLD_FRAMES = 5        # frames que se mantiene la caja de matricula si no se detecta

# Parámetros visuales y de OCR de matrículas
COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35

# Parámetros del OCR sobre vídeo
OCR_CADA = 5
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 6

model, postprocessor = load_rtdetr(family="v2", size="l", device=DEVICE)

Cargando RT-DETR familia 'v2', tamaño 'l' en cuda
Load PResNet50 state_dict


**TRACKING Y DETECCIÓN RT-DETR**

Las funciones del procesamiento de vídeo las utiliza `procesar_video_rtdetr`:
- **`preprocess`**: Redimensiona el fotograma a 640x640 y lo convierte en tensor, una sola vez por fotograma.
- **`make_tracker`**: Crea un tracker ByteTrack con el tiempo de memoria indicado.
- **`letterbox`**: Redimensiona el recorte manteniendo la proporción y rellena con gris, para no deformar las matrículas.
- **`run_detector`**: Ejecuta el detector y filtra por confianza y clases.
- **`detect_plates_in_crops`**: Busca la mejor matrícula dentro del recorte de cada vehículo y la devuelve en coordenadas del fotograma.

**PROCESAMIENTO DE VÍDEO**

Se recorre el vídeo fotograma a fotograma: se detectan personas y vehículos, se les asigna un identificador, se busca la matrícula de cada vehículo y se lee con el VLM. El bucle completo está en `procesar_video_rtdetr`. Se guardan el vídeo anotado `resultado.mp4` y un CSV con todos los resultados `resultado.csv` en `RT-DETR_VLM_result`.

In [ ]:
path_test = 'C0142.MP4'
output_dir = os.path.abspath('RT-DETR_VLM_result')

plates_seen, lecturas = procesar_video_rtdetr(
    path_test, output_dir, model, postprocessor, model_plate, postprocessor_plate,
    DEVICE, leer_placa, CLASS_NAMES, BASE_COLORS,
    plate_classes=PLATE_CLASS,
    conf_vehiculos=0.5,
    plate_conf=PLATE_CONF, hold_frames=HOLD_FRAMES, max_crops=MAX_CROPS,
    min_crop_side=MIN_CROP_SIDE, crop_margin=CROP_MARGIN,
    color_placa=COLOR_PLACA, escala_texto_placa=ESCALA_TEXTO_PLACA,
    ocr_cada=OCR_CADA, min_ancho_ocr=MIN_ANCHO_OCR, min_chars_placa=MIN_CHARS_PLACA,
    ocr_score_ok=OCR_SCORE_OK, ocr_max_intentos=OCR_MAX_INTENTOS,
)

**EVALUACIÓN DEL VLM SOBRE EL DATASET**

Se evalúa el conjunto detector de matrículas + VLM sobre las imágenes anotadas, comparando contra el texto real. Se obtienen:
- **Exact match**: Proporción de matrículas leídas exactamente igual que la anotación.
- **Acierto por carácter**: Similitud media basada en la distancia de Levenshtein.
- **Techo del OCR**: Lectura directa del recorte de la anotación, sin depender del detector.
- **Lectura cruda vs. corregida**: La lectura tal cual y tras la corrección por posición.
- **Tiempos**: Milisegundos por lectura (media y mediana), con una lectura de calentamiento previa.

La función `evaluar_dataset` está en `utils_matriculas.py` y es la misma en todos los cuadernos: es independiente del detector, que se le pasa como una función `detectar(img) -> [(caja_xyxy, confianza)]`.

Se reutiliza el VLM ya cargado (`leer_placa`) y el detector de matrículas (`model_plate`). `crear_detector_rtdetr` (`utils_matriculas.py`) aplica **letterbox** y devuelve las cajas en píxeles originales.

In [ ]:
# ---------- Configuración de la evaluación ----------
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"

MAX_IMAGENES = 200 
CONF_EVAL = 0.35
IOU_MATCH = 0.5
PAD = 0.05
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("vlm_eval")

# cargar_gt (utils) recibe la carpeta de anotaciones y las claves como argumentos: las fijamos aqui
gt_dataset = partial(cargar_gt, anotaciones_dir=ANOTACIONES, gt_keys=GT_KEYS)

# Detector RT-DETR de matrículas con letterbox: imagen BGR -> [(caja_xyxy, confianza), ...]
detectar_rtdetr = crear_detector_rtdetr(model_plate, postprocessor_plate, DEVICE,
                                        CONF_EVAL, PLATE_CLASS)


# ---------- Ejecución de la evaluación ----------
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista, detectar_rtdetr, leer_placa, gt_dataset, SALIDA, "errores-rtdetr",
                    "rtdetr_resultados_vlm.csv", "RT-DETR", iou_match=IOU_MATCH, pad=PAD,
                    gt_keys=GT_KEYS, delimiter=";")
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset (variable BASE).")


Imágenes evaluadas: 200 | placas anotadas: 411 (con texto: 411) | detecciones RT-DETR: 620

--- RESUMEN DE EXACTITUD OCR (lectura cruda) ---
OCR sobre detecciones de RT-DETR       n=201   exact match=0.811 | por carácter=0.929 | 760 ms de media (515 de mediana)
OCR sobre recorte anotado (techo)      n=208   exact match=0.788 | por carácter=0.924 | 862 ms de media (514 de mediana)

--- CON CORRECCIÓN POR POSICIÓN ---
OCR + corrección, detecciones RT-DETR  n=201   exact match=0.851 | por carácter=0.939 | 760 ms de media (515 de mediana)
OCR + corrección, recorte anotado      n=208   exact match=0.822 | por carácter=0.932 | 862 ms de media (514 de mediana)

Placas evaluadas encontradas por RT-DETR: 201/411
Falsos positivos geométricos (IoU < 0.5): 330
Detecciones en placas anotadas sin texto (None): 89

CSV: vlm_eval\rtdetr_resultados_vlm.csv
Recortes con error: vlm_eval\errores-rtdetr


In [11]:
import utils_matriculas as um

resultados_rtdetr = um.evaluar_detector(
    detector=detectar_rtdetr,
    imagenes_dir=IMAGENES,
    anotaciones_dir=ANOTACIONES,
    salida_dir=Path("resultados") / "RTDETR_VLM",
    leer_ocr=leer_placa,
    nombre_modelo="RTDETR_VLM",
    max_imagenes=MAX_IMAGENES,
    iou_match=IOU_MATCH,
    pad=PAD,
    limpiar_salida=True
)


Modelo: RTDETR_VLM
Imágenes: 200
[1/200] 00000.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[2/200] 00001.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[3/200] 00002.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[4/200] 00003.jpg | GT: 1 | Detectadas: 1 | Fallos: 0
[5/200] 0173FVB-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[6/200] 0224CXR-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[7/200] 0270DRN.JPG | GT: 2 | Detectadas: 2 | Fallos: 0
[8/200] 0404HCF-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[9/200] 0617KFZ.JPG | GT: 2 | Detectadas: 2 | Fallos: 0
[10/200] 0696CMB.JPG | GT: 2 | Detectadas: 2 | Fallos: 0
[11/200] 0848GLN-cgg.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[12/200] 09d2d922-35f1-4ed3-93a5-82a1ed54d8cb.JPG | GT: 1 | Detectadas: 1 | Fallos: 0
[13/200] 1 (1).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[14/200] 1 (10).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[15/200] 1 (11).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[16/200] 1 (12).jpeg | GT: 1 | Detectadas: 1 | Fallos: 0
[17/200] 1 (13).jpeg | GT: 